In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import glob
import os

data_path = os.path.join('..', 'data')
csv_files = glob.glob(os.path.join(data_path, '*_clean.csv'))

df_list = []
for file in csv_files:
    df_temp = pd.read_csv(file, parse_dates=['Date'])
    df_list.append(df_temp)

all_countries = pd.concat(df_list, ignore_index=True)

print(f"Loaded {len(csv_files)} countries: {all_countries['Country'].unique()}")
print(f"Total rows: {len(all_countries)}")
print(f"Date range: {all_countries['Date'].min()} to {all_countries['Date'].max()}")
all_countries.head()

In [ ]:
monthly_temp = all_countries.set_index('Date').groupby('Country').resample('M')['T2M'].mean().reset_index()

plt.figure(figsize=(14, 7))
for country in monthly_temp['Country'].unique():
    country_data = monthly_temp[monthly_temp['Country'] == country]
    plt.plot(country_data['Date'], country_data['T2M'], label=country, linewidth=1.5)

plt.xlabel('Date')
plt.ylabel('Mean Temperature (°C)')
plt.title('Monthly Average Temperature by Country (2015-2026)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
temp_summary = all_countries.groupby('Country')['T2M'].agg(['mean', 'median', 'std']).round(2)
temp_summary.columns = ['Mean T2M (°C)', 'Median T2M (°C)', 'Std Dev T2M (°C)']
temp_summary = temp_summary.sort_values('Mean T2M (°C)', ascending=False)

print("Temperature Summary by Country:")
display(temp_summary)

In [ ]:
plt.figure(figsize=(12, 6))
all_countries.boxplot(column='PRECTOTCORR', by='Country', patch_artist=True)
plt.title('Precipitation Distribution by Country')
plt.suptitle('')
plt.ylabel('Precipitation (mm/day)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
rain_summary = all_countries.groupby('Country')['PRECTOTCORR'].agg(['mean', 'median', 'std']).round(3)
rain_summary.columns = ['Mean Rain (mm/day)', 'Median Rain (mm/day)', 'Std Dev Rain (mm/day)']
rain_summary = rain_summary.sort_values('Std Dev Rain (mm/day)', ascending=False)

print("Precipitation Summary by Country:")
display(rain_summary)

In [ ]:
country_groups = [group['T2M'].dropna().values for name, group in all_countries.groupby('Country')['T2M']]
f_stat, p_value = stats.f_oneway(*country_groups)

print(f"ANOVA Results for T2M across all five countries:")
print(f"  F-statistic: {f_stat:.3f}")
print(f"  P-value: {p_value:.6f}")
print(f"  Significant difference? {'Yes (p < 0.05)' if p_value < 0.05 else 'No (p >= 0.05)'}")

In [ ]:
heat_days = all_countries[all_countries['T2M_MAX'] > 35].groupby(['Country', 'Year']).size().unstack(fill_value=0)

heat_days.T.plot(kind='bar', figsize=(12, 6))
plt.title('Days per Year with T2M_MAX > 35°C (Extreme Heat)')
plt.ylabel('Number of Days')
plt.xlabel('Year')
plt.legend(title='Country')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
def max_consecutive_dry(series):
    is_dry = series < 1
    return is_dry.groupby((is_dry != is_dry.shift()).cumsum()).sum().max()

dry_spells = all_countries.groupby(['Country', 'Year'])['PRECTOTCORR'].apply(max_consecutive_dry).unstack()

dry_spells.T.plot(kind='bar', figsize=(12, 6))
plt.title('Maximum Consecutive Dry Days (< 1mm) per Year')
plt.ylabel('Max Consecutive Dry Days')
plt.xlabel('Year')
plt.legend(title='Country')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
## Climate Vulnerability Ranking

| Rank | Country | Warming Trend | Rainfall Stability | Extreme Heat | Overall Risk |
|------|---------|---------------|-------------------|--------------|--------------|
| 1 | [Country] | [Fastest/Moderate] | [Very Low/Low] | [Highest] | Most Critical |
| 2 | [Country] | [Rising/Stable] | [Moderate] | [High] | High |
| 3 | [Country] | [Stable] | [Moderate] | [Moderate] | Moderate |
| 4 | [Country] | [Stable] | [High] | [Low] | Lower |
| 5 | [Country] | [Stable] | [High] | [Lowest] | Lowest |

## 5 Key Findings for COP32

### 1. Fastest Warming: [Country]
- **What is changing:** [Country] shows a warming trend of [X]°C/year, with mean temperatures reaching [Y]°C
- **What it caused:** Rising temperatures threaten crop yields and livestock productivity in this agricultural economy
- **What it demands:** Urgent adaptation finance for heat-resilient crops and early warning systems

### 2. Most Unstable Precipitation: [Country]
- **What is changing:** [Country] has the highest rainfall variability (std dev: [X] mm/day)
- **What it caused:** Erratic rainfall undermines agricultural planning and food security
- **What it demands:** Investment in water infrastructure and index-based crop insurance

### 3. Extreme Events Are Intensifying
- **What is changing:** [Country] experiences [X] extreme heat days per year; [Country] has dry spells up to [Y] days
- **What it caused:** Compound heat-drought events amplify each other, depleting soil moisture
- **What it demands:** Loss and damage mechanisms must recognize compound extreme events

### 4. Ethiopia's Relative Position
- **What is changing:** Ethiopia shows [more stable/moderate] climate patterns compared to neighbors
- **What it caused:** This stability positions Ethiopia as a credible regional leader
- **What it demands:** Ethiopia should champion regional cooperation on transboundary water and early warning systems

### 5. Priority Country for Climate Finance: [Country]
- **What is changing:** [Country] ranks highest on composite vulnerability index
- **What it caused:** Converging climate stressors threaten development gains and could displace millions
- **What it demands:** Priority access to the Green Climate Fund with measurable resilience outcomes